## Import libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib


## Load the dataset

In [2]:
df = pd.read_csv("../data/processed/cleaned_customer_churn.csv")

In [3]:
# Check the dataset
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## Remove irrelevant columns
CustormerID contains unique customer identification numbers. It does not provide meaningful information about customer behaviour for our prediction task.

In [5]:
df.drop("customerID", axis=1, inplace=True)

In [6]:
# Confirm that it has been removed
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Separate features (X) and target variable (y)

In [7]:
X = df.drop("Churn", axis=1)

y = df["Churn"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (7043, 19)
Target shape: (7043,)


## Categorical Encoding

In [8]:
# Identify categorical columns
categorical_cols = X.select_dtypes(
    include="object"
).columns

print(categorical_cols)
print("Number of categorical columns:", len(categorical_cols))

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')
Number of categorical columns: 15


In [9]:
# Encode categorical variables
X_encoded = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

In [10]:
# Check the encoded dataset 
X_encoded.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,1,0,0,1,0,...,0,0,0,0,0,0,1,0,1,0
1,0,34,56.95,1889.50,1,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,1
2,0,2,53.85,108.15,1,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,1
3,0,45,42.30,1840.75,1,0,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
4,0,2,70.70,151.65,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0


In [11]:
print("Original features:", X.shape)
print("Encoded features:", X_encoded.shape)

Original features: (7043, 19)
Encoded features: (7043, 30)


In [12]:
X_encoded.dtypes.value_counts()

int32      26
int64       2
float64     2
Name: count, dtype: int64

### Categorical Encoding Summary

One-Hot Encoding was applied to the 15 categorical features using `pd.get_dummies()` with `drop_first=True`.

The encoding transformed the original dataset from 19 features to 30 numerical features while retaining all 7,043 customer records.

The resulting dataset contains integer and floating-point data types, making it suitable for machine learning algorithms that require numerical inputs.

**Conclusion:** Categorical encoding was successfully completed, converting the categorical variables into numerical representations for model development.

## Train-Test Split

In [13]:
# Encode target variable y from Yes/No into 1/0.
y = y.map({"No": 0, "Yes": 1})

y.value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [14]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [15]:
# Check the results
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (5634, 30)
X_test: (1409, 30)
y_train: (5634,)
y_test: (1409,)


### Train-Test Split Summary

The dataset was divided into training and testing sets using an 80:20 ratio.

- **Training Set:** Contains 5,634 records (80%) and 30 features, used to train the machine learning model.
- **Testing Set:** Contains 1,409 records (20%) and 30 features, used to evaluate the model's performance on unseen data.

The `stratify=y` parameter was applied to maintain the original distribution of churn and non-churn customers in both datasets. Additionally, `random_state=42` was used to ensure reproducibility.

**Conclusion:** The train-test split was successfully completed, providing separate datasets for model training and evaluation while preserving the target variable distribution.

## Feature Scaling


In [16]:

scaler = StandardScaler()

numerical_cols = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_cols] = scaler.fit_transform(
    X_train[numerical_cols]
)

X_test_scaled[numerical_cols] = scaler.transform(
    X_test[numerical_cols]
)

In [17]:
# Verify the scaling
X_train_scaled[numerical_cols].describe()

,tenure,MonthlyCharges,TotalCharges
count,5.634000e+03,5.634000e+03,5.634000e+03
mean,-1.008935e-17,-2.402527e-16,2.522338e-17
std,1.000089e+00,1.000089e+00,1.000089e+00
min,-1.322329e+00,-1.544028e+00,-1.008922e+00
25%,-9.559779e-01,-9.711977e-01,-8.321009e-01
50%,-1.418632e-01,1.848336e-01,-3.968446e-01
75%,9.164859e-01,8.319124e-01,6.741944e-01
max,1.608483e+00,1.785939e+00,2.801869e+00


### Feature Scaling Summary

Feature scaling was performed using **StandardScaler** to standardise the numerical variables and ensure consistency in their scales before model development.

- **Selected Features:** Scaling was applied to `tenure`, `MonthlyCharges`, and `TotalCharges`.
- **Training Data:** The scaler was fitted and applied to the training set using `fit_transform()`, calculating the mean and standard deviation required for standardisation.
- **Testing Data:** The fitted scaler was applied to the testing set using `transform()`, ensuring that the same scaling parameters were used across both datasets.
- **Data Preservation:** Copies of the original training and testing datasets were maintained to preserve the unscaled values.

After scaling, the numerical features had a mean approximately equal to 0 and a standard deviation approximately equal to 1.

**Conclusion:** Feature scaling was successfully completed using StandardScaler. By fitting the scaler exclusively on the training data and applying the learned parameters to the testing data, data leakage was avoided. The resulting datasets were prepared for subsequent machine learning model development and evaluation.




## Save dataset for model building

In [18]:
prepared_data = {
    "X_train": X_train_scaled,
    "X_test": X_test_scaled,
    "y_train": y_train,
    "y_test": y_test
}

joblib.dump(prepared_data, "prepared_data.pkl")

['prepared_data.pkl']

## Save the scaler

In [20]:
joblib.dump(scaler, "standard_scaler.pkl")

['standard_scaler.pkl']